# 01 — Data Engineering & Preprocessing (Amazon Reviews, UBCF) — Phase 1

## Purpose & Research Context
**Problem:** Amazon Reviews represent a highly sparse user-item interaction space. Only ~0.001–0.01% of possible user-item pairs have ratings. This sparsity severely limits classical collaborative filtering's ability to compute reliable user similarity.

**Solution in this phase:** Transform raw JSONL interactions into feature matrices suitable for both classical and quantum clustering approaches.

## Data Pipeline
**Input (from `QACF/AmazonReviews/raw/`):**
- `Appliances.json` — review records with user, item (ASIN), rating, review text, helpfulness signals
- `meta_Appliances.json` — Item metadata (titles, categories, description snippets)

**Why these files matter:**
- Review text and metadata provide redundancy when collaborative signal is sparse
- Explicit ratings (1–5 stars) give numerical interaction strength
- Helpfulness signals hint at which reviews users trust (important for sparse neighborhoods)

## Processing Steps

### 1. **Chunked ID Discovery & Statistics**
- Parse JSONL in 200K-record chunks (memory-efficient handling)
- Extract reviewerID → user, ASIN → item, rating (1–5 scale)
- Build sets of unique users and items
- Compute review length statistics (avg., std., percentiles) to understand text richness
  - *Why:* Sparse users may have very short or absent review text; this metric tracks data quality

### 2. **Sparse Matrix Construction (CSR format)**
- Build user×item rating matrix
- Shape: `(n_users, n_items)`, CSR (compressed sparse row) for efficient row slicing
- Typical shape: ~100K users × ~1.2M items with ~10M ratings
- **Sparsity metric:** density ≈ 10M/(100K × 1.2M) ≈ 0.0000083 (< 0.001% filled)
  - *Why CSR matrices:* Required for fast neighbor retrieval in collaborative filtering; standard in recommendation systems

### 3. **Dimensionality Reduction (Truncated SVD)**
- Apply truncated SVD to user×item matrix: 32 latent dimensions
- Captures primary collaborative patterns without overfitting to noise
- Produces:
  - `user_latent.npy` — User representations (100K × 32)
  - `item_latent.npy` — Item representations (1.2M × 32)
  - Explained variance: typically 35–50% of total (sufficient for downstream tasks)
  - *Why SVD:* Reduces sparsity's negative impact by projecting into dense latent space; standard denoising technique

### 4. **Content Feature Engineering**
- Extract review text TF-IDF vectors from reviews for each item (up to 1,200 chars per item)
- Extract product category one-hot + hierarchical features
- Produces:
  - `content_features.npz` — Item content (1.2M × 4600 features, sparse)
  - *Why content features:* For extreme cold-start items with <5 ratings, content-based similarity is the only signal

### 5. **Combined Feature Spaces**
- Concatenate latent + content features for hybrid routing
- Produces:
  - `user_combined.npy` — User features (100K × 32 latent + aux)
  - `item_combined.npy` — Item features (1.2M × 32 latent + 4600 content)
  - *Why combined:* Allows hybrid recommender to switch between collaborative and content signals based on sparsity profile

## Key Outputs
| Artifact | Shape | Purpose |
|----------|-------|---------|
| `user_sparse.npz` | (100K, 1.2M) sparse | Raw interaction matrix |
| `user_latent.npy` | (100K, 32) dense | Denoised user signals |
| `item_latent.npy` | (1.2M, 32) dense | Denoised item signals |
| `content_features.npz` | (1.2M, 4600) sparse | Fallback for cold items |
| `user_combined.npy` | (100K, 32+) dense | User features for quantum kernel |
| `id_maps.pkl` | dict | Maps review IDs ↔ array indices |
| `category_sparsity.csv` | stats | Sparsity by category (diagnostic) |

## Connection to Research Claim
This phase creates the feature space that both **classical K-Means** (Phase 2) and **quantum kernel clustering** (Phase 3) will operate on. By normalizing sparsity effects into latent/content representations, we create comparable baselines for the quantum-vs-classical comparison.

**Key hypothesis tested downstream:** Quantum clustering on these latent features will be more robust to user sparsity than classical K-Means because quantum kernels can capture non-Euclidean neighborhood structure in sparse regimes.

In [4]:
from pathlib import Path
from collections import defaultdict
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

START_DIR = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in [START_DIR, *START_DIR.parents] if (p / "data").exists() and (p / "QACF").exists()),
    START_DIR,
)

RAW_CANDIDATES = [
    Path(r"C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\data"),
    START_DIR / "data",
    START_DIR / "FashionReviews" / "data",
    START_DIR / "Fashion" / "data",
    START_DIR / "AmazonReviews" / "All_Fashion" / "data",
    START_DIR / "QACF" / "AmazonReviews" / "FashionReviews" / "data",
    START_DIR / "QACF" / "AmazonReviews" / "Fashion" / "data",
    START_DIR.parent / "FashionReviews" / "data",
    START_DIR.parent / "Fashion" / "data",
    START_DIR.parent / "AmazonReviews" / "FashionReviews" / "data",
    START_DIR.parent / "AmazonReviews" / "Fashion" / "data",
    PROJECT_DIR / "QACF" / "AmazonReviews" / "FashionReviews" / "data",
    PROJECT_DIR / "QACF" / "AmazonReviews" / "Fashion" / "data",
]

raw_dir = next((p for p in RAW_CANDIDATES if p.exists()), None)
if raw_dir is None:
    raise FileNotFoundError("Could not locate Fashion data directory.")

review_candidates = ["Fashion.csv", "All_Fashion.csv", "All_Fashion.json", "Fashion.json"]
meta_candidates = ["meta_All_Fashion.json", "meta_Fashion.json", "metadata.json"]

REVIEWS_PATH = next((raw_dir / name for name in review_candidates if (raw_dir / name).exists()), None)
META_PATH = next((raw_dir / name for name in meta_candidates if (raw_dir / name).exists()), None)
if REVIEWS_PATH is None:
    available = sorted([p.name for p in raw_dir.glob("*") if p.is_file()])
    raise FileNotFoundError(f"No review data file found in {raw_dir}. Found files: {available}")


def _clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = []
    for c in df.columns:
        if isinstance(c, str):
            cleaned.append(c.strip())
        else:
            cleaned.append(c)
    out = df.copy()
    out.columns = cleaned
    return out


def _has_known_review_cols(columns) -> bool:
    cols = {str(c).strip().lower() for c in columns}
    known = {
        "reviewerid", "user_id", "user",
        "asin", "parent_asin", "item_id", "product_id",
        "overall", "rating", "stars", "score",
        "reviewtext", "text", "review", "review_body",
    }
    return len(cols.intersection(known)) > 0


def _normalize_headerless_review_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    df = chunk.copy()

    numeric_cache = {}
    for c in df.columns:
        numeric_cache[c] = pd.to_numeric(df[c], errors="coerce")

    rating_col = None
    rating_score = -1.0
    for c, num in numeric_cache.items():
        valid = num.notna().mean()
        in_range = ((num >= 0.5) & (num <= 5.5)).mean()
        score = float(valid * in_range)
        if score > rating_score and score > 0.6:
            rating_score = score
            rating_col = c

    unix_col = None
    unix_score = -1.0
    for c, num in numeric_cache.items():
        if c == rating_col:
            continue
        valid = num.notna().mean()
        in_range = ((num >= 900_000_000) & (num <= 2_200_000_000)).mean()
        score = float(valid * in_range)
        if score > unix_score and score > 0.5:
            unix_score = score
            unix_col = c

    object_cols = [c for c in df.columns if df[c].dtype == object]
    user_col = None
    for c in object_cols:
        if c == rating_col or c == unix_col:
            continue
        series = df[c].astype(str)
        avg_len = series.str.len().mean()
        if avg_len >= 6:
            user_col = c
            break

    remaining = [c for c in df.columns if c not in {rating_col, unix_col, user_col}]

    text_col = None
    for c in remaining:
        series = df[c].astype(str)
        if series.str.len().mean() > 20:
            text_col = c
            break

    item_col = None
    for c in remaining:
        if c != text_col:
            item_col = c
            break

    rename_map = {}
    if user_col is not None:
        rename_map[user_col] = "reviewerID"
    if item_col is not None:
        rename_map[item_col] = "asin"
    if rating_col is not None:
        rename_map[rating_col] = "overall"
    if text_col is not None:
        rename_map[text_col] = "reviewText"

    return df.rename(columns=rename_map)


def _normalize_review_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    df = _clean_columns(chunk)
    cols_lower = {str(c).lower(): c for c in df.columns}

    rename_map = {}

    user_aliases = ["reviewerid", "user_id", "user"]
    item_aliases = ["asin", "parent_asin", "item_id", "product_id"]
    rating_aliases = ["overall", "rating", "stars", "score"]
    text_aliases = ["reviewtext", "text", "review", "review_body"]

    for alias in user_aliases:
        if alias in cols_lower:
            rename_map[cols_lower[alias]] = "reviewerID"
            break
    for alias in item_aliases:
        if alias in cols_lower:
            rename_map[cols_lower[alias]] = "asin"
            break
    for alias in rating_aliases:
        if alias in cols_lower:
            rename_map[cols_lower[alias]] = "overall"
            break
    for alias in text_aliases:
        if alias in cols_lower:
            rename_map[cols_lower[alias]] = "reviewText"
            break

    df = df.rename(columns=rename_map)

    required_present = all(col in df.columns for col in ["reviewerID", "asin", "overall"])
    if not required_present and all(isinstance(c, int) for c in df.columns):
        df = _normalize_headerless_review_chunk(df)

    return df


def iter_review_chunks(path: Path, chunksize: int):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        reader = pd.read_csv(path, chunksize=chunksize)
        try:
            first = next(reader)
        except StopIteration:
            return

        if _has_known_review_cols(first.columns):
            yield _normalize_review_chunk(first)
            for chunk in reader:
                yield _normalize_review_chunk(chunk)
        else:
            for chunk in pd.read_csv(path, chunksize=chunksize, header=None):
                yield _normalize_review_chunk(chunk)

    elif suffix in {".json", ".jsonl"}:
        for chunk in pd.read_json(path, lines=True, chunksize=chunksize):
            yield _normalize_review_chunk(chunk)
    else:
        raise ValueError(f"Unsupported review file extension: {suffix}. Expected .csv/.json/.jsonl")


def iter_meta_chunks(path: Path, chunksize: int):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        for chunk in pd.read_csv(path, chunksize=chunksize):
            yield _clean_columns(chunk)
    elif suffix in {".json", ".jsonl"}:
        for chunk in pd.read_json(path, lines=True, chunksize=chunksize):
            yield _clean_columns(chunk)
    else:
        raise ValueError(f"Unsupported metadata file extension: {suffix}. Expected .csv/.json/.jsonl")


OUT_DIR = PROJECT_DIR / "data" / "processed_Fashion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 200_000
N_COMPONENTS = 32
SEED = 42
MAX_REVIEW_TFIDF = 4000
MAX_CATEGORY_TFIDF = 600

rng = np.random.default_rng(SEED)

print(f"start dir: {START_DIR}")
print(f"project dir: {PROJECT_DIR}")
print(f"raw dir: {raw_dir}")
print(f"reviews: {REVIEWS_PATH}")
print(f"metadata: {META_PATH}")
print(f"output dir: {OUT_DIR}")

start dir: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\notebooks
project dir: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\notebooks
raw dir: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\data
reviews: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\data\Fashion.csv
metadata: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\data\meta_Fashion.json
output dir: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\FashionReviews\notebooks\data\processed_Fashion


In [5]:
# Chunked pass for ID discovery + explicit rating/review-text extraction
all_users, all_items = set(), set()
n_rows = 0

review_len_count = 0
review_len_sum = 0.0
review_len_sum2 = 0.0
review_len_min = np.inf
review_len_max = 0
review_len_sample = []

for chunk in iter_review_chunks(REVIEWS_PATH, CHUNK_SIZE):
    user_col = "reviewerID" if "reviewerID" in chunk.columns else ("user_id" if "user_id" in chunk.columns else None)
    item_col = "asin" if "asin" in chunk.columns else ("parent_asin" if "parent_asin" in chunk.columns else None)
    rating_col = "overall" if "overall" in chunk.columns else ("rating" if "rating" in chunk.columns else None)
    text_col = "reviewText" if "reviewText" in chunk.columns else ("text" if "text" in chunk.columns else None)

    if user_col is None or item_col is None or rating_col is None:
        raise ValueError(
            "Missing required review columns. Need user/rating/item from "
            f"['reviewerID'|'user_id'], ['overall'|'rating'], ['asin'|'parent_asin']; got {set(chunk.columns)}"
        )

    cols = [user_col, item_col, rating_col] + ([text_col] if text_col else [])
    chunk = chunk[cols].dropna(subset=[user_col, item_col, rating_col]).copy()
    if text_col is None:
        chunk["reviewText"] = ""
    else:
        chunk["reviewText"] = chunk[text_col].fillna("").astype(str)

    chunk = chunk.rename(columns={user_col: "user_raw", item_col: "item_raw", rating_col: "rating"})
    chunk["user_raw"] = chunk["user_raw"].astype(str)
    chunk["item_raw"] = chunk["item_raw"].astype(str)
    chunk["rating"] = chunk["rating"].astype(np.float32)

    n_rows += len(chunk)
    all_users.update(chunk["user_raw"].unique().tolist())
    all_items.update(chunk["item_raw"].unique().tolist())

    lengths = chunk["reviewText"].str.split().map(len).to_numpy(dtype=np.int32)
    if len(lengths) > 0:
        review_len_count += len(lengths)
        review_len_sum += float(lengths.sum())
        review_len_sum2 += float((lengths.astype(np.float64) ** 2).sum())
        review_len_min = min(review_len_min, int(lengths.min()))
        review_len_max = max(review_len_max, int(lengths.max()))

        sample_n = min(2000, len(lengths))
        sample_idx = rng.choice(len(lengths), size=sample_n, replace=False)
        review_len_sample.extend(lengths[sample_idx].tolist())

orig_users = np.array(sorted(all_users), dtype=object)
orig_items = np.array(sorted(all_items), dtype=object)

user_to_idx = {u: i for i, u in enumerate(orig_users.tolist())}
item_to_idx = {m: i for i, m in enumerate(orig_items.tolist())}

unique_users = np.arange(len(orig_users), dtype=np.int64)
unique_movies = np.arange(len(orig_items), dtype=np.int64)

review_len_mean = review_len_sum / max(review_len_count, 1)
review_len_var = max(review_len_sum2 / max(review_len_count, 1) - review_len_mean ** 2, 0.0)
review_len_std = float(np.sqrt(review_len_var))
review_len_sample_arr = np.array(review_len_sample, dtype=np.float32) if review_len_sample else np.array([0], dtype=np.float32)

review_length_stats_df = pd.DataFrame(
    [
        {
            "n_reviews": int(review_len_count),
            "mean_words": float(review_len_mean),
            "std_words": review_len_std,
            "min_words": int(0 if review_len_min == np.inf else review_len_min),
            "max_words": int(review_len_max),
            "sample_p50_words": float(np.percentile(review_len_sample_arr, 50)),
            "sample_p90_words": float(np.percentile(review_len_sample_arr, 90)),
            "sample_p99_words": float(np.percentile(review_len_sample_arr, 99)),
        }
    ]
)

print(f"rows={n_rows:,} users={len(orig_users):,} items={len(orig_items):,}")
print("Explicit ratings + review text extracted from chunked reviews")
print(review_length_stats_df.T)

rows=883,636 users=186,189 items=749,233
Explicit ratings + review text extracted from chunked reviews
                         0
n_reviews         883636.0
mean_words             0.0
std_words              0.0
min_words              0.0
max_words              0.0
sample_p50_words       0.0
sample_p90_words       0.0
sample_p99_words       0.0


In [6]:
# Build user-focused CSR matrix: rows=users, cols=items
row_parts, col_parts, val_parts = [], [], []

for chunk in iter_review_chunks(REVIEWS_PATH, CHUNK_SIZE):
    user_col = "reviewerID" if "reviewerID" in chunk.columns else ("user_id" if "user_id" in chunk.columns else None)
    item_col = "asin" if "asin" in chunk.columns else ("parent_asin" if "parent_asin" in chunk.columns else None)
    rating_col = "overall" if "overall" in chunk.columns else ("rating" if "rating" in chunk.columns else None)

    if user_col is None or item_col is None or rating_col is None:
        raise ValueError(
            "Missing required review columns for matrix build. Need user/rating/item from "
            f"['reviewerID'|'user_id'], ['overall'|'rating'], ['asin'|'parent_asin']; got {set(chunk.columns)}"
        )

    chunk = chunk[[user_col, item_col, rating_col]].dropna().copy()
    chunk[user_col] = chunk[user_col].astype(str)
    chunk[item_col] = chunk[item_col].astype(str)

    row_parts.append(chunk[user_col].map(user_to_idx).to_numpy(dtype=np.int32, copy=False))
    col_parts.append(chunk[item_col].map(item_to_idx).to_numpy(dtype=np.int32, copy=False))
    val_parts.append(chunk[rating_col].to_numpy(dtype=np.float32, copy=False))

rows = np.concatenate(row_parts)
cols = np.concatenate(col_parts)
vals = np.concatenate(val_parts)

user_sparse = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(unique_users), len(unique_movies)),
    dtype=np.float32,
)

n_components = min(N_COMPONENTS, max(2, min(user_sparse.shape) - 1))
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
user_latent = svd.fit_transform(user_sparse).astype(np.float32)
item_latent = svd.components_.T.astype(np.float32)

density = user_sparse.nnz / (user_sparse.shape[0] * user_sparse.shape[1])
print(f"user_sparse shape={user_sparse.shape}, nnz={user_sparse.nnz:,}, density={density:.6f}")
print(f"user_latent shape={user_latent.shape}")
print(f"item_latent shape={item_latent.shape}")
print(f"explained_variance_ratio_sum={svd.explained_variance_ratio_.sum():.4f}")

user_sparse shape=(186189, 749233), nnz=875,121, density=0.000006
user_latent shape=(186189, 32)
item_latent shape=(749233, 32)
explained_variance_ratio_sum=0.0981


In [7]:
# Content features from review text + product categories, plus Amazon-specific analyses
import time

FAST_MODE = True
MAX_REVIEW_TEXT_CHARS_PER_ITEM = 1200 if FAST_MODE else 3000
MAX_REVIEW_TFIDF_LOCAL = min(MAX_REVIEW_TFIDF, 1200) if FAST_MODE else MAX_REVIEW_TFIDF
MAX_CATEGORY_TFIDF_LOCAL = min(MAX_CATEGORY_TFIDF, 300) if FAST_MODE else MAX_CATEGORY_TFIDF
MAX_REVIEW_CHUNKS_FOR_TEXT = 40 if FAST_MODE else None
PROGRESS_EVERY = 5

item_review_buffers = {}
t_text_start = time.perf_counter()
processed_review_chunks = 0

for chunk_idx, chunk in enumerate(iter_review_chunks(REVIEWS_PATH, CHUNK_SIZE), start=1):
    if MAX_REVIEW_CHUNKS_FOR_TEXT is not None and chunk_idx > MAX_REVIEW_CHUNKS_FOR_TEXT:
        break

    processed_review_chunks += 1
    item_col = "asin" if "asin" in chunk.columns else ("parent_asin" if "parent_asin" in chunk.columns else None)
    text_col = "reviewText" if "reviewText" in chunk.columns else ("text" if "text" in chunk.columns else None)

    if item_col is None:
        continue

    if text_col is None:
        chunk = chunk[[item_col]].copy()
        chunk["review_text"] = ""
    else:
        chunk = chunk[[item_col, text_col]].copy()
        chunk = chunk.rename(columns={text_col: "review_text"})

    chunk[item_col] = chunk[item_col].astype(str)
    chunk["review_text"] = chunk["review_text"].fillna("").astype(str)
    chunk = chunk[chunk[item_col].isin(item_to_idx)]

    grouped = chunk.groupby(item_col)["review_text"].apply(lambda s: " ".join([x for x in s if x]))
    for asin, text in grouped.items():
        if not text:
            continue

        current = item_review_buffers.get(asin, "")
        remaining = MAX_REVIEW_TEXT_CHARS_PER_ITEM - len(current)
        if remaining <= 0:
            continue

        snippet = text[:remaining]
        item_review_buffers[asin] = (current + " " + snippet).strip()

    if chunk_idx % PROGRESS_EVERY == 0:
        elapsed = time.perf_counter() - t_text_start
        print(f"Review text pass: chunk {chunk_idx:,} | items with text={len(item_review_buffers):,} | elapsed={elapsed/60:.1f} min")

has_review_text = any(bool(item_review_buffers.get(asin, "").strip()) for asin in orig_items)
if has_review_text:
    review_vec = TfidfVectorizer(
        max_features=MAX_REVIEW_TFIDF_LOCAL,
        min_df=2,
        stop_words="english",
        ngram_range=(1, 2),
    )
    item_docs_iter = (item_review_buffers.get(asin, "") for asin in orig_items)
    review_tfidf = review_vec.fit_transform(item_docs_iter).astype(np.float32)
else:
    review_tfidf = csr_matrix((len(orig_items), 1), dtype=np.float32)

meta_map = {}
def flatten_categories(value):
    if isinstance(value, list):
        out = []
        for v in value:
            if isinstance(v, list):
                out.extend([str(x).strip() for x in v if str(x).strip()])
            elif pd.notna(v):
                s = str(v).strip()
                if s:
                    out.append(s)
        return out
    if pd.isna(value):
        return []
    s = str(value).strip()
    return [s] if s else []

if META_PATH is not None and META_PATH.exists():
    for chunk in iter_meta_chunks(META_PATH, CHUNK_SIZE):
        meta_item_col = "asin" if "asin" in chunk.columns else ("parent_asin" if "parent_asin" in chunk.columns else None)
        if meta_item_col is None:
            continue

        if "categories" in chunk.columns:
            cat_series = chunk["categories"]
        elif "category" in chunk.columns:
            cat_series = chunk["category"]
        else:
            cat_series = pd.Series([[]] * len(chunk), index=chunk.index)

        keep = pd.DataFrame({"asin": chunk[meta_item_col].astype(str), "cats": cat_series})
        keep = keep[keep["asin"].isin(item_to_idx)]

        for asin, cats in zip(keep["asin"], keep["cats"]):
            parsed = flatten_categories(cats)
            if parsed:
                meta_map[asin] = parsed

category_docs = [" ".join(meta_map.get(asin, ["unknown_category"])) for asin in orig_items]
cat_vec = TfidfVectorizer(max_features=MAX_CATEGORY_TFIDF_LOCAL, token_pattern=r"(?u)\b[\w\-]+\b")
category_tfidf = cat_vec.fit_transform(category_docs).astype(np.float32)

content_features = hstack([review_tfidf, category_tfidf], format="csr", dtype=np.float32)

content_components_cap = 16 if FAST_MODE else 32
content_components = min(content_components_cap, max(2, min(content_features.shape[0], content_features.shape[1]) - 1))
content_svd = TruncatedSVD(n_components=content_components, random_state=SEED)
item_content_latent = content_svd.fit_transform(content_features).astype(np.float32)

user_activity = np.asarray((user_sparse != 0).sum(axis=1)).ravel().astype(np.float32)
user_content_latent = user_sparse.dot(item_content_latent)
user_content_latent = user_content_latent / np.maximum(user_activity[:, None], 1.0)
user_content_latent = user_content_latent.astype(np.float32)

user_combined = np.hstack([user_latent, user_content_latent]).astype(np.float32)
item_combined = np.hstack([item_latent, item_content_latent]).astype(np.float32)

primary_category = np.array([meta_map.get(asin, ["unknown_category"])[0] for asin in orig_items], dtype=object)
cat_df = pd.DataFrame({"item_idx": np.arange(len(orig_items), dtype=np.int32), "category": primary_category})

rows = []
for category, sub in cat_df.groupby("category"):
    item_idx = sub["item_idx"].to_numpy(dtype=np.int32)
    if len(item_idx) == 0:
        continue
    submat = user_sparse[:, item_idx]
    denom = user_sparse.shape[0] * len(item_idx)
    density = float(submat.nnz / denom) if denom > 0 else 0.0
    rows.append(
        {
            "category": category,
            "n_items": int(len(item_idx)),
            "nnz": int(submat.nnz),
            "density": density,
            "sparsity": 1.0 - density,
            "mean_rating": float(submat.data.mean()) if submat.nnz > 0 else np.nan,
        }
    )

category_sparsity_df = pd.DataFrame(rows).sort_values(["n_items", "density"], ascending=[False, False])

save_npz(OUT_DIR / "user_sparse.npz", user_sparse)
np.save(OUT_DIR / "user_latent.npy", user_latent)
np.save(OUT_DIR / "item_latent.npy", item_latent)
save_npz(OUT_DIR / "content_features.npz", content_features)
np.save(OUT_DIR / "item_content_latent.npy", item_content_latent)
np.save(OUT_DIR / "user_combined.npy", user_combined)
np.save(OUT_DIR / "item_combined.npy", item_combined)
category_sparsity_df.to_csv(OUT_DIR / "category_sparsity.csv", index=False)
review_length_stats_df.to_csv(OUT_DIR / "review_length_stats.csv", index=False)

with open(OUT_DIR / "id_maps.pkl", "wb") as f:
    pickle.dump(
        {
            "unique_users": unique_users,
            "unique_movies": unique_movies,
            "user_to_idx": user_to_idx,
            "movie_to_idx": item_to_idx,
            "orig_user_ids": orig_users,
            "orig_item_ids": orig_items,
            "primary_category": primary_category,
        },
        f,
    )

print(f"review_tfidf shape: {review_tfidf.shape}")
print(f"category_tfidf shape: {category_tfidf.shape}")
print(f"content_features shape: {content_features.shape}")
print(f"user_combined shape: {user_combined.shape}")
print(f"item_combined shape: {item_combined.shape}")
print(f"FAST_MODE: {FAST_MODE}")
print(f"review chunks processed for text: {processed_review_chunks}")
print(f"review text cap per item (chars): {MAX_REVIEW_TEXT_CHARS_PER_ITEM}")
print(f"review tfidf max_features used: {MAX_REVIEW_TFIDF_LOCAL}")
print(f"category tfidf max_features used: {MAX_CATEGORY_TFIDF_LOCAL}")
print("Saved: user_sparse.npz, user_latent.npy, item_latent.npy, content_features.npz,")
print("       item_content_latent.npy, user_combined.npy, item_combined.npy,")
print("       id_maps.pkl, category_sparsity.csv, review_length_stats.csv")
print("Top category sparsity rows:")
print(category_sparsity_df.head(10))

Review text pass: chunk 5 | items with text=0 | elapsed=0.1 min


c:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\.quant\lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var


review_tfidf shape: (749233, 1)
category_tfidf shape: (749233, 1)
content_features shape: (749233, 2)
user_combined shape: (186189, 34)
item_combined shape: (749233, 34)
FAST_MODE: True
review chunks processed for text: 5
review text cap per item (chars): 1200
review tfidf max_features used: 1200
category tfidf max_features used: 300
Saved: user_sparse.npz, user_latent.npy, item_latent.npy, content_features.npz,
       item_content_latent.npy, user_combined.npy, item_combined.npy,
       id_maps.pkl, category_sparsity.csv, review_length_stats.csv
Top category sparsity rows:
           category  n_items     nnz   density  sparsity  mean_rating
0  unknown_category   749233  875121  0.000006  0.999994     3.944955


## D-Wave Local Feature Selection (Simulator)

D-Wave annealing for feature selection in sparse user data (local sim now, real Leap later per supervisor request).

This section formulates a binary QUBO with one variable per latent feature ($z_i \in \{0,1\}$), where selected features maximize a variance-based proxy while enforcing a feature-budget penalty.

Objective (BQM proxy):
- Minimize $-\sum_i w_i z_i + \lambda(\sum_i z_i-k)^2$
- $w_i$: normalized variance contribution of feature $i$
- $k$: target number of selected latent features

In [8]:
# D-Wave Local Feature Selection (Simulator)
from pathlib import Path
import numpy as np

use_dwave_sim = True

try:
    import dimod
except ImportError as exc:
    raise ImportError(
        "dimod is required for local D-Wave-style simulation. Install with: pip install dimod"
    ) from exc

if "OUT_DIR" not in globals():
    OUT_DIR = Path("data/processed_Fashion")
if "SEED" not in globals():
    SEED = 42

if "user_latent" not in globals():
    user_latent = np.load(OUT_DIR / "user_latent.npy")

X_latent = np.asarray(user_latent, dtype=np.float32)
n_users, n_features = X_latent.shape

# Target selected dimensionality (for 32 latent features, default keeps half)
TARGET_FEATURES = min(16, n_features)
TARGET_FEATURES = max(1, TARGET_FEATURES)

# Proxy objective weights: explained-variance approximation from per-feature variance
feature_var = X_latent.var(axis=0).astype(np.float64)
if np.allclose(feature_var.sum(), 0.0):
    feature_scores = np.ones(n_features, dtype=np.float64) / n_features
else:
    feature_scores = feature_var / feature_var.sum()

# Binary QUBO (BQM): minimize -variance_selected + cardinality penalty
k = int(TARGET_FEATURES)
lambda_cardinality = float(np.max(feature_scores) * 2.0 + 1e-9)

linear = {
    i: float(lambda_cardinality * (1 - 2 * k) - feature_scores[i])
    for i in range(n_features)
}
quadratic = {
    (i, j): float(2.0 * lambda_cardinality)
    for i in range(n_features)
    for j in range(i + 1, n_features)
}
offset = float(lambda_cardinality * (k ** 2))

bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset, vartype=dimod.BINARY)

if use_dwave_sim:
    if n_features <= 20:
        sampler = dimod.ExactSolver()
        sample_set = sampler.sample(bqm)
    else:
        sampler = dimod.SimulatedAnnealingSampler()
        sample_set = sampler.sample(bqm, num_reads=50)
else:
    # Upgrade path for real D-Wave Leap later:
    # sampler = LeapHybridCQMSampler(token=os.getenv('DWAVE_API_TOKEN'))
    raise NotImplementedError("Set use_dwave_sim=True until D-Wave Leap token is available.")

best = sample_set.first.sample
selected_mask = np.array([1 if best[i] == 1 else 0 for i in range(n_features)], dtype=np.int8)

# Enforce exact-k fallback for stable downstream dimensions
if selected_mask.sum() != k:
    top_idx = np.argsort(feature_scores)[-k:]
    selected_mask[:] = 0
    selected_mask[top_idx] = 1

user_latent_selected = X_latent[:, selected_mask.astype(bool)]

np.save(OUT_DIR / "user_latent_selected.npy", user_latent_selected.astype(np.float32))
np.save(OUT_DIR / "selected_mask.npy", selected_mask)

retained_ratio = float(feature_var[selected_mask.astype(bool)].sum() / (feature_var.sum() + 1e-12))
print(f"D-Wave local sim complete | n_features={n_features}, selected={int(selected_mask.sum())}")
print(f"Approx retained variance ratio: {retained_ratio:.4f}")
print("Saved: user_latent_selected.npy, selected_mask.npy")

# Small-slice test (200 users) as quick sanity check
slice_n = min(200, n_users)
X_slice = X_latent[:slice_n]
X_slice_selected = X_slice[:, selected_mask.astype(bool)]
print(f"Slice test OK | input={X_slice.shape}, selected={X_slice_selected.shape}")

D-Wave local sim complete | n_features=32, selected=16
Approx retained variance ratio: 0.6204
Saved: user_latent_selected.npy, selected_mask.npy
Slice test OK | input=(200, 32), selected=(200, 16)
